### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from groq import Groq
load_dotenv()
GROQ_API_KEY = os.getenv("GROQKEY")
client = Groq(
    api_key=GROQ_API_KEY,
)

c:\Users\owais\OneDrive\Desktop\rag\ragpractice\envs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### LLM Inference

In [2]:
def groq_llm(system_prompt,user_input,llm="openai/gpt-oss-120b",temp=0.5,strm=False,reasoning="medium"):
    chat_completion = client.chat.completions.create(
    messages=[
        { 
            "role": "user",
            "content": user_input,
        }, 
        {"role": "system", "content": system_prompt},
    ],
    model=llm,
    temperature=temp,
    stream=strm,
    reasoning_effort=reasoning,
)

    return chat_completion.choices[0].message.content
groq_llm(system_prompt="ur helpful assistant",user_input="what is the true meaning of life?")

'The short answer is: **there isn’t a single, universally‑agreed‑upon “true meaning of life.”**  \nWhat gives life meaning is something each person (or culture, tradition, or community) discovers for themselves, often by weaving together a mix of philosophy, religion, personal experience, and the way they relate to the world around them.\n\nBelow is a quick tour of the most common ways people have tried to answer the question, followed by some practical ideas for exploring your own sense of purpose.\n\n---\n\n## 1. Philosophical Perspectives  \n\n| School of thought | Core idea about meaning | Key thinkers |\n|-------------------|------------------------|--------------|\n| **Existentialism** | Life has no pre‑written purpose; we create meaning through our choices and authentic action. | Jean‑Paul Sartre, Albert Camus, Simone de Beauvoir |\n| **Absurdism** | The universe is indifferent, but we can embrace the “absurd” and find joy in the struggle itself. | Albert Camus |\n| **Stoicism**

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory): # defines a function to process all PDFs in a directory and its parameter can be called anything
    """Process all PDF files in the specified directory."""
    all_documents = [] # Initialize an empty list to store all document objects (pages returned by the loader)
    pdf_dir = Path(pdf_directory) 
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files.")
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            # Load the PDF file
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f" loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
    # Process all PDFs in the data directory

all_pdf_documents = process_all_pdfs("../envs/data/pdf")
all_pdf_documents

Found 3 PDF files.
Processing file: ..\envs\data\pdf\401 Homework 5.pdf
 loaded 6 pages
Processing file: ..\envs\data\pdf\introtocandp.pdf
 loaded 30 pages
Processing file: ..\envs\data\pdf\sample-local-pdf.pdf
 loaded 3 pages

Total documents loaded: 39
 loaded 30 pages
Processing file: ..\envs\data\pdf\sample-local-pdf.pdf
 loaded 3 pages

Total documents loaded: 39


[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'file_path': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '401 Homework 5', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '401 Homework 5.pdf', 'file_type': 'pdf'}, page_content='Q1:\nFor each of the two questions below, decide whether the answer is (i) “Yes,” (ii) “No,” or (iii)\n“Unknown, because it would resolve the question of whether P = NP.” Give a brief explanation of\nyour answer.\n(a)\nIs it the case that Interval Scheduling ≤\x00 Vertex Cover?\nAnswer: (i) Yes.\nThe decision version of the Interval Scheduling Problem asks whether there exists a subset of\nat least k non-overlapping intervals from a given collection. This problem is known to be\nsolvable in polynomial time using a greedy 

In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'file_path': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '401 Homework 5', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '401 Homework 5.pdf', 'file_type': 'pdf'}, page_content='Q1:\nFor each of the two questions below, decide whether the answer is (i) “Yes,” (ii) “No,” or (iii)\n“Unknown, because it would resolve the question of whether P = NP.” Give a brief explanation of\nyour answer.\n(a)\nIs it the case that Interval Scheduling ≤\x00 Vertex Cover?\nAnswer: (i) Yes.\nThe decision version of the Interval Scheduling Problem asks whether there exists a subset of\nat least k non-overlapping intervals from a given collection. This problem is known to be\nsolvable in polynomial time using a greedy 

In [5]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 39 documents into 105 chunks.

Example chunk:
Content: Q1:
For each of the two questions below, decide whether the answer is (i) “Yes,” (ii) “No,” or (iii)
“Unknown, because it would resolve the question of whether P = NP.” Give a brief explanation of
you...
Metadata: {'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'file_path': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '401 Homework 5', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '401 Homework 5.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'file_path': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '401 Homework 5', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '401 Homework 5.pdf', 'file_type': 'pdf'}, page_content='Q1:\nFor each of the two questions below, decide whether the answer is (i) “Yes,” (ii) “No,” or (iii)\n“Unknown, because it would resolve the question of whether P = NP.” Give a brief explanation of\nyour answer.\n(a)\nIs it the case that Interval Scheduling ≤\x00 Vertex Cover?\nAnswer: (i) Yes.\nThe decision version of the Interval Scheduling Problem asks whether there exists a subset of\nat least k non-overlapping intervals from a given collection. This problem is known to be\nsolvable in polynomial time using a greedy 

In [7]:
keyword = "I digress."
found = [doc for doc in chunks if keyword.lower() in doc.page_content.lower()]
print(len(found))


1


### embedding and vectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [27]:
import chromadb
print(f"ChromaDB version: {chromadb.__version__}")

ChromaDB version: 1.2.1


In [9]:
class EmbeddingManager:
    """Manages document generation using SentenceTransformer"""

    def __init__(self, model_name: str = "paraphrase-MiniLM-L3-v2"):
        """Initialize the EmbeddingManager
        Args:
            model_name: Huggingface model name for sentence embeddings
        """
        self.model_name = model_name #Object attribute value respectively
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts: List of strings to generate embeddings for
        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model is not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ## Intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: paraphrase-MiniLM-L3-v2
Model loaded successfully. Embedding dimension: 384
Model loaded successfully. Embedding dimension: 384


### VectorStore

In [28]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = r"c:\Users\owais\OneDrive\Desktop\rag\ragpractice\data\vector_store"):
        """
        Initialize the VectorStore
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
                embedding_function=None
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()})")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
            documents: List of document objects with 'page_content' and 'metadata'
            embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)  # Copy existing metadata
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document context
            documents_text.append(doc.page_content)

            # Embedding
            embedding_list.append(embedding.tolist())

        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection now: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [18]:
chunks

[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'file_path': '..\\envs\\data\\pdf\\401 Homework 5.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '401 Homework 5', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': '401 Homework 5.pdf', 'file_type': 'pdf'}, page_content='Q1:\nFor each of the two questions below, decide whether the answer is (i) “Yes,” (ii) “No,” or (iii)\n“Unknown, because it would resolve the question of whether P = NP.” Give a brief explanation of\nyour answer.\n(a)\nIs it the case that Interval Scheduling ≤\x00 Vertex Cover?\nAnswer: (i) Yes.\nThe decision version of the Interval Scheduling Problem asks whether there exists a subset of\nat least k non-overlapping intervals from a given collection. This problem is known to be\nsolvable in polynomial time using a greedy 

In [29]:
### Convert the text into embeddings
texts = [doc.page_content for doc in chunks]

## Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore = VectorStore()
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 105 texts...


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.93it/s]



Generated embeddings with shape: (105, 384)
Vector store initialized with collection: pdf_documents
Existing documents in collection: 0)
Adding 105 documents to the vector store...
Successfully added 105 documents to the vector store.
Total documents in collection now: 105


### Retriever Pipeline From VectorStore

In [30]:
class RAGretriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the Retriever
        Args:
            vector_store: Instance of VectorStore
            embedding_manager: Instance of EmbeddingManager
            top_k: Number of top documents to retrieve
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for the given query
        Args:
            query: User query string
        Returns:
            List of retrieved document metadata and content
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results["documents"][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distances': distances,
                            'rank': i + 1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found.")
        
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGretriever(vectorstore, embedding_manager)

In [44]:
rag_retriever

In [49]:
# run the raw query and print the first batch of documents (no filtering)
results = vectorstore.collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
print("documents (first batch):", results.get("documents", [None])[0])
print("ids (first batch):", results.get("ids", [None])[0])
print("distances (first batch):", results.get("distances", [None])[0])

documents (first batch): ['- Let {Sᵢ₁, Sᵢ₂, ..., Sᵢₖ} be a set cover of U.\n- Hire counselors i₁, i₂, ..., iₖ.\n- Since their subsets cover U, their qualifications cover all sports.\n(<-) From Efficient Recruiting to Set Cover:\n- Let {i₁, i₂, ..., iₜ} (with t ≤k) be the hired counselors.\n- The corresponding subsets Sᵢ₁, Sᵢ₂, ..., Sᵢₜ cover U.\n- Therefore, these subsets form a set cover of size at most k.\nSince we have a polynomial-time reduction from Set Cover to Efficient Recruiting and the\nproblem is in NP, the Efficient Recruiting Problem is NP-complete.\nQ5:\nProve that Hitting Set is NP-complete.\nAnswer:\nWe will prove that the Hitting Set Problem is NP-complete by showing that:\n1. It is in NP.\n2. It is NP-hard by reducing the NP-complete Vertex Cover Problem to it.\n(1) Hitting Set is in NP:\nGiven a subset H ⊆A of size at most k, we can verify in polynomial time whether H is a hitting\nset:\n- For each subset Bᵢ, check whether H ∩Bᵢ≠∅.\n- This verification takes O(m · k)

In [ ]:
print(vectorstore.collection.count())


735


In [10]:
import os
vector_store_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), "data", "vector_store"))
print(f"Vector store absolute path: {vector_store_path}")

Vector store absolute path: c:\Users\owais\OneDrive\Desktop\rag\ragpractice\data\vector_store


In [32]:
self.collection = self.client.get_or_create_collection(
    name=self.collection_name,
    metadata={"description": "PDF document embeddings for RAG"},
    embedding_function=None,
    # 👇 ensures cosine similarity is used
    metadata_config={"hnsw:space": "cosine"}
)


NameError: name 'self' is not defined

In [16]:
import shutil

# Delete the vector store directory
vector_store_path = os.path.join(os.path.dirname(os.getcwd()), "data", "vector_store")
if os.path.exists(vector_store_path):
    shutil.rmtree(vector_store_path)
    print(f"Deleted vector store at: {vector_store_path}")
else:
    print("Vector store directory doesn't exist")

Deleted vector store at: c:\Users\owais\OneDrive\Desktop\rag\ragpractice\data\vector_store
